# IBM Telco Customer Churn Prediction
## End-to-End Supervised Machine Learning Project
**Dataset:** IBM Telco Customer Churn
**Objective:** Reproducible comparison of baseline and optimized classification models


## 1. Problem Definition
The application domain is telecommunications customer retention. The business problem is to identify customers at risk of churn so that retention resources can be targeted before customers leave. The prediction objective is binary classification of `Churn Value` (1 = churn, 0 = no churn). Predictors describe demographics, subscribed services, contract and billing arrangements, tenure, charges, and customer lifetime value. Machine learning is appropriate because churn depends on multiple interacting customer characteristics and past labelled outcomes are available for supervised learning.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
DATA_PATH = 'Telco_customer_churn.xlsx'

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name='Telco_Churn', engine='openpyxl')
print('Dimensions:', df.shape)
display(df.head())

## 2. Exploratory Data Analysis

In [ ]:
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique()}))
display(df.describe(include='all').T)
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
print('Missing values after numeric conversion:')
display(df.isna().sum().sort_values(ascending=False).head(10))
print(df['Churn Value'].value_counts())
print(df['Churn Value'].value_counts(normalize=True).rename('proportion'))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
sns.countplot(data=df, x='Churn Label', ax=axes[0,0])
sns.boxplot(data=df, x='Churn Label', y='Tenure Months', ax=axes[0,1])
sns.boxplot(data=df, x='Churn Label', y='Monthly Charges', ax=axes[0,2])
sns.countplot(data=df, x='Contract', hue='Churn Label', ax=axes[1,0])
sns.countplot(data=df, x='Internet Service', hue='Churn Label', ax=axes[1,1])
sns.heatmap(df[['Tenure Months','Monthly Charges','Total Charges','CLTV','Churn Value']].corr(), annot=True, cmap='vlag', center=0, ax=axes[1,2])
for ax in axes.ravel(): ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

### EDA interpretation
The target is moderately imbalanced, with the non-churn class substantially larger than the churn class. Churn is more prevalent among month-to-month customers, customers with shorter tenure, and customers using fiber optic service. Monthly charges also differ by churn status. Correlation is used only for numeric variables and does not replace examination of categorical relationships.

## 3. Data Preprocessing

In [ ]:
target = 'Churn Value'
# Remove direct leakage, post-outcome information, identifiers, constants, and high-cardinality location fields.
leakage = ['Churn Label','Churn Score','Churn Reason']
identifiers = ['CustomerID','Count','Country','State','City','Zip Code','Lat Long','Latitude','Longitude']
X = df.drop(columns=identifiers + leakage + [target])
y = df[target].astype(int)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])
print(X_train.shape, X_test.shape, num_cols, len(cat_cols))

**Justification.** Median and most-frequent imputation are fitted only on the training data through a pipeline. One-hot encoding converts categorical fields without imposing an artificial order. Standardization supports logistic regression. A stratified 80/20 split preserves the class ratio. The identical preprocessing object and split are used for all four models.

## 4. Baseline Models

In [ ]:
baseline_lr = Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
baseline_dt = Pipeline([('preprocessor', preprocessor), ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE))])
baseline_lr.fit(X_train, y_train); baseline_dt.fit(X_train, y_train)

## 5. Hyperparameter Tuning and Optimized Models

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lr_search = GridSearchCV(Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]), {'classifier__C':[0.1,1], 'classifier__class_weight':[None,'balanced'], 'classifier__solver':['liblinear']}, scoring='f1', cv=cv, n_jobs=1)
dt_search = GridSearchCV(Pipeline([('preprocessor', preprocessor), ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE))]), {'classifier__max_depth':[4,8], 'classifier__min_samples_split':[2,20], 'classifier__min_samples_leaf':[1,10], 'classifier__criterion':['gini'], 'classifier__class_weight':[None,'balanced']}, scoring='f1', cv=cv, n_jobs=1)
lr_search.fit(X_train,y_train); dt_search.fit(X_train,y_train)
print('Best LR:', lr_search.best_params_, lr_search.best_score_)
print('Best DT:', dt_search.best_params_, dt_search.best_score_)

## 6. Training/Test Evaluation, Classification Reports, and Confusion Matrices

In [ ]:
models = {'Baseline Logistic Regression':baseline_lr, 'Baseline Decision Tree':baseline_dt, 'Optimized Logistic Regression':lr_search.best_estimator_, 'Optimized Decision Tree':dt_search.best_estimator_}
rows=[]
fig, axes = plt.subplots(2,2,figsize=(11,8))
for ax,(name,model) in zip(axes.ravel(),models.items()):
    train_pred=model.predict(X_train); test_pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
    rows.append({'Model':name,'Train Accuracy':accuracy_score(y_train,train_pred),'Test Accuracy':accuracy_score(y_test,test_pred),'Precision':precision_score(y_test,test_pred),'Recall':recall_score(y_test,test_pred),'F1-score':f1_score(y_test,test_pred),'ROC-AUC':roc_auc_score(y_test,prob)})
    print('\n',name); print(classification_report(y_test,test_pred,target_names=['No Churn','Churn']))
    sns.heatmap(confusion_matrix(y_test,test_pred),annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax); ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()
comparison=pd.DataFrame(rows).sort_values('F1-score',ascending=False).reset_index(drop=True)
display(comparison.style.format({c:'{:.3f}' for c in comparison.columns if c!='Model'}))

## 7. Discussion Questions
1. Which model achieved the best test-set F1-score and why is that important for churn?
2. Which model had the smallest train-test accuracy gap?
3. Did tuning improve each algorithm relative to its baseline?
4. What precision-recall trade-off is acceptable for a retention campaign?
5. How would intervention cost and customer lifetime value change the classification threshold?
6. What evidence of bias or performance differences across customer groups should be assessed before deployment?

## 8. Conclusion and Deployment Recommendations
Select the final model using test-set evidence, with particular attention to churn recall, precision, F1-score, ROC-AUC, and the train-test gap. Before production deployment, validate on a later time period, tune the probability threshold using business costs, assess subgroup fairness, package the full preprocessing-model pipeline, monitor data drift and performance, and establish periodic retraining and rollback procedures. Churn Score, Churn Label, and Churn Reason must not be used as predictors because they would leak information about the outcome.

## Source
IBM Telco Customer Churn Dataset: https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset?utm_source=chatgpt.com

*Note: Results are reproducible with `random_state=42`; exact values may vary across library versions.*